## Lecture 12 - Binomial regression

McElreath's lecture for today: https://www.youtube.com/watch?v=hRJtKCIDTwc

McElreath's lectures for the whole book are available here: https://github.com/rmcelreath/statrethinking_winter2019

An R/Stan repo of code is available here: https://vincentarelbundock.github.io/rethinking2/

An excellent port to Python/PyMC Code is available here: https://github.com/dustinstansbury/statistical-rethinking-2023

You are encouraged to work through both of these versions to re-enforce what we're doing in class.

In [ ]:
# Load R packages
library(cmdstanr)    # R interface to Stan
library(posterior)   # Working with posterior draws
library(bayesplot)   # Plotting posterior draws
library(loo)         # WAIC and PSIS-LOO cross-validation
library(ggplot2)     # bayesplot figures are ggplots, so we can add to them
library(dagitty)     # Drawing DAGs


# Helper functions
stdize <- function(x) (x - mean(x))/sd(x)


indexall <- function(L){
    poo <- unique(L)
    Ix <- as.integer(factor(L, levels = poo))
    list(poo, Ix)
}

# Stan's sampler with the same defaults as pm.sample(1000): 4 chains, 1000 warmup, 1000 draws
sample_stan <- function(model, data){
    model$sample(data = data, chains = 4, parallel_chains = 4, iter_warmup = 1000,
                 iter_sampling = 1000, refresh = 0, show_exceptions = FALSE)
}

# Save the current figure to file (uncomment the savefig() calls below to use)
savefig <- function(file, width = 7, height = 5){
    dev.copy(jpeg, file, width = width, height = height, units = "in", res = 300)
    invisible(dev.off())
}

## Monkey hands

In chapter 11 (p325) there is a description of experiments done by [Silk et al 2005](https://www.nature.com/articles/nature04243), whereby chimpanzees are given the option of having food for themselves alone, or having the same quantity of food for themselves plus giving it to another chimp at the opposite end of a table where, given this particular setup, pulling the right hand lever would give just the subject (the *actor* in the text) a grape, while pulling the left lever would also give the other chimp a grape. The placement of the other grape is randomly assigned because chimps are also right/left handed. 

In [ ]:
# Grab chimp data
cdata <- read.csv('chimpanzees.csv', sep = ';')
# Display top 5 rows
head(cdata, 5)

Here we have four potential treatments:

- prosoc_left = 0 AND condition = 0 (Two food items on right and no partner)
- prosoc_left = 0 AND condition = 1 (Two food items on right and partner present)
- prosoc_left = 1 AND condition = 0 (Two food items on left and no partner)
- prosoc_left = 1 AND condition = 1 (Two food items on left and partner present)

that we can encode into one of four treatments:

In [ ]:
cdata$treatment <- 1 + cdata$prosoc_left + 2*cdata$condition
cdata$treatment

To figure out if chimps enact pro-social behaviour and give the other chimp a grape (without penalty to themselves) reuquires a statistical model. To start, we'll build a simple model based on what happens with the left-hand lever (i.e. if the left was pulled it's 1, if the right it's 0):

$$
\begin{aligned}
L_i & \sim Bin(1,p_i) \\
logit(p_i) & = \beta_{actor}+\beta_{treatment}
\end{aligned}
$$

The next thing we need to do is specify some priors. And for that we'll start with some reasonable values then take a look through piror predictive simulation. Thinking about this, we know that the full range of 0 to 1, is about -4 to 4 on the log-odds (logit) scale. So for a normal prior to span that range, we'd need somthing that is about $2SD=4$ or $SD=2$. So let's try that:

In [ ]:
# Inverse-logit function (R also has this built in, as plogis())
invlogit <- function(x) exp(x)/(1 + exp(x))

# Random samples from a N(0,2)
hist(invlogit(rnorm(1000, 0, 2)), main = "", xlab = "")
# savefig('histo1.jpg')

Not quite flat - McElreath uses $N(0, 1.5)$, so let's try that

In [ ]:
# Random samples from a N(0,1.5)
hist(invlogit(rnorm(10000, 0, 1.5)), main = "", xlab = "")
# savefig('histo2.jpg')

Can we do better still? $N(0, 1.7)$?

In [ ]:
# Random samples from a N(0,1.7)
hist(invlogit(rnorm(10000, 0, 1.7)), main = "", xlab = "")
# savefig('histo3.jpg')

For the treatment effects, what we want to know is the difference in how often the left hand lever is pulled when there is a pro-social grape in the partner's box and when there isn't. So this we can represent as leading to some sort of increase in probability. In general these kinds of behavioural difference in psychology are low, so we can pick a prior with low potential differences, something like $N(0,0.5)$:

In [ ]:
n_ <- 10000
# Baseline log-odds, then two treatment levels drawn from the prior
b0_ <- rnorm(n_, 0, 1.7)
nt <- invlogit(b0_ + rnorm(n_, 0, 0.5))
tn <- invlogit(b0_ + rnorm(n_, 0, 0.5))
hist(abs(nt - tn), main = "", xlab = "")
# savefig('betaT.jpg')

In [ ]:
# Average difference between treatments, a priori
mean(abs(nt - tn))

This isn't too bad, it favours lower values with an average of about a 10% difference. This might seem too narrow but remember that any normal prior can still take on extreeme values (they're not prohibited), but they're just unlikely. If you have a lot of data this prior will be overwhealmed. Keep in mind what such regularization does: **good priors hurt the fit to sample but improve prediction**. 

Ok, so let's build our model:

In [ ]:
# Grab data
# Left pull - response
L <- cdata$pulled_left
# Individual chimps
tmp <- indexall(cdata$actor)
Actor <- tmp[[1]]; Ia <- tmp[[2]]
Chimp <- paste('Chimp', Actor)
nchimps <- length(Actor)
# Treatment
tmp <- indexall(cdata$treatment)
Treatment <- tmp[[1]]; It <- tmp[[2]]
Treatment <- c('R/N', 'L/N', 'R/P', 'L/P')
ntreat <- length(Treatment)

In [ ]:
Treatment

In PyMC the linear model is wrapped in `pm.invlogit()` before being handed to the `Binomial` likelihood. Stan builds the inverse-link into its likelihoods instead, so here we hand `bernoulli_logit()` the linear model on the **logit scale** and it does the back-transform internally - this is both faster and numerically safer than taking the inverse-logit ourselves. The indexed intercepts (PyMC's `dims='Chimp'` and `dims='Treat'`) become plain vectors indexed by an integer array - remembering that R and Stan count from 1, not 0.

In [ ]:
Chimps_code <- "
data {
  int<lower=0> N;                              // Number of observations
  int<lower=1> nchimps;
  int<lower=1> ntreat;
  array[N] int<lower=1, upper=nchimps> Ia;     // Chimp index
  array[N] int<lower=1, upper=ntreat> It;      // Treatment index
  array[N] int<lower=0, upper=1> L;            // Left pull - response
}
parameters {
  vector[nchimps] Actor;        // Individual intercepts
  vector[ntreat] Treatment;     // Treatment effects
}
model {
  // Individual intercepts
  Actor ~ normal(0, 1.7);
  // Treatment effects
  Treatment ~ normal(0, 0.5);

  // Linear model and likelihood
  L ~ bernoulli_logit(Actor[Ia] + Treatment[It]);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = bernoulli_logit_lpmf(L[i] | Actor[Ia[i]] + Treatment[It[i]]);
}
"
Chimps <- cmdstan_model(write_stan_file(Chimps_code))

In [ ]:
trace_c <- sample_stan(Chimps, list(N = length(L), nchimps = nchimps, ntreat = ntreat,
                                    Ia = Ia, It = It, L = L))

In [ ]:
tmp <- trace_c$summary(c("Actor", "Treatment"))
tmp$Label <- c(Chimp, Treatment)
tmp

In [ ]:
ChimpTrace <- trace_c

In [ ]:
# bayesplot has no transform argument, so put the draws on the probability scale first
Actor_p <- plogis(ChimpTrace$draws("Actor", format = "matrix"))
colnames(Actor_p) <- Chimp
mcmc_intervals(Actor_p)
# ggsave('m1forest.jpg', dpi = 300)

From this, we can see that various chimps have various levels of handedness, with one chimp only ever pulling the left lever. 

Let's move on to the treatments:


In [ ]:
Treat_draws <- ChimpTrace$draws("Treatment", format = "matrix")
colnames(Treat_draws) <- Treatment
mcmc_intervals(Treat_draws) + vline_0()
# ggsave('m1forest2.jpg', dpi = 300)

From this we can see that, while there are slight differences between R/N vs R/P and L/N vs L/P, they're pretty small, with considerable overlap for L and some overlap in R. We can also look at this in terms of absolute differences:


In [ ]:
# Grab posteriors as a matrix: rows = iterations, columns = treatments
tmp <- trace_c$draws("Treatment", format = "matrix")

In [ ]:
hist(tmp[, 1] - tmp[, 3], xlab = "",
     main = paste0('Right P(effect)=R/N-R/P=', sum((tmp[, 1] - tmp[, 3]) > 0)/nrow(tmp)))
abline(v = 0, col = 'red', lwd = 4)
# savefig('m1RT.jpg')

In [ ]:
hist(tmp[, 2] - tmp[, 4], xlab = "",
     main = paste0('Left P(effect)=L/N-L/P=', sum((tmp[, 2] - tmp[, 4]) > 0)/nrow(tmp)))
abline(v = 0, col = 'red', lwd = 4)
# savefig('m1LT.jpg')

So both have more density above zero, suggestive of an effect, with stronger evidence among right handers. However these differences are N-P (no one there - another chimp there) which means chimps are slighly less likey to pull the left or right levers when there is someone on the other side. Jerks.

To see what our model actually does here, we can push our estimates back through the model and output the expectations for each chimp; first a look at the data itself:

In [ ]:
# Set figure size
options(repr.plot.width = 10, repr.plot.height = 3)
plot(NA, xlim = c(0.5, 28.5), ylim = c(0, 1.15), xlab = "", ylab = "", xaxt = 'n')
oset <- 1
for (i in 1:7){
    # Grab chimp
    l <- L[Ia == i]
    t <- It[Ia == i]
    # Grab data for each treatment
    rn <- mean(l[t == 1])
    ln <- mean(l[t == 2])
    rp <- mean(l[t == 3])
    lp <- mean(l[t == 4])
    # Plot rights
    lines(c(oset, oset + 2), c(rn, rp), col = 'blue')
    points(oset, rn, col = 'blue', bg = 'white', pch = 21)
    points(oset + 2, rp, col = 'blue', bg = 'blue', pch = 21)
    # Plot lefts
    lines(c(oset + 1, oset + 3), c(ln, lp), col = 'blue')
    points(oset + 1, ln, col = 'blue', bg = 'white', pch = 21)
    points(oset + 3, lp, col = 'blue', bg = 'blue', pch = 21)
    oset <- oset + 4
}
for (x in c(4, 8, 12, 16, 20, 24)) abline(v = x + 0.5, col = 'grey')
for (x in c(4, 8, 12, 16, 20, 24, 28)) text(x - 3 + 0.5, 1.12, paste('Actor', x/4))
abline(h = 0.5, lty = 2, col = 'grey')
legend('bottomright', legend = c('Alone', 'Partner'), pch = 21, col = 'blue',
       pt.bg = c('white', 'blue'), bty = 'n', horiz = TRUE)
# savefig('data.jpg', width = 10, height = 3)
options(repr.plot.width = 7, repr.plot.height = 5)

Next, what the model actually sees:

In [ ]:
a_trace <- trace_c$draws("Actor", format = "matrix")
t_trace <- trace_c$draws("Treatment", format = "matrix")

In [ ]:
# Grab effects sizes for each treatment
t1 <- t_trace[, 1]
t2 <- t_trace[, 2]
t3 <- t_trace[, 3]
t4 <- t_trace[, 4]

# Set figure size
options(repr.plot.width = 10, repr.plot.height = 3)
plot(NA, xlim = c(0.5, 28.5), ylim = c(0, 1.15), xlab = "", ylab = "", xaxt = 'n')
oset <- 1
for (i in 1:7){
    # Grab chimp intercept
    l <- a_trace[, i]
    # Calculate individual effects
    rn <- median(invlogit(l + t1))
    rn2 <- sd(invlogit(l + t1))*2
    ln <- median(invlogit(l + t2))
    ln2 <- sd(invlogit(l + t2))*2
    rp <- median(invlogit(l + t3))
    rp2 <- sd(invlogit(l + t3))*2
    lp <- median(invlogit(l + t4))
    lp2 <- sd(invlogit(l + t4))*2

    # Plot rights
    lines(c(oset, oset + 2), c(rn, rp), col = 'blue')
    lines(c(oset, oset), c(rn - rn2, rn + rn2), col = 'black')
    lines(c(oset + 2, oset + 2), c(rp - rp2, rp + rp2), col = 'black')

    points(oset, rn, col = 'blue', bg = 'white', pch = 21)
    points(oset + 2, rp, col = 'blue', bg = 'blue', pch = 21)

    # Plot lefts
    lines(c(oset + 1, oset + 3), c(ln, lp), col = 'blue')
    lines(c(oset + 1, oset + 1), c(ln - ln2, ln + ln2), col = 'black')
    lines(c(oset + 3, oset + 3), c(lp - lp2, lp + lp2), col = 'black')
    points(oset + 1, ln, col = 'blue', bg = 'white', pch = 21)
    points(oset + 3, lp, col = 'blue', bg = 'blue', pch = 21)
    oset <- oset + 4
}
for (x in c(4, 8, 12, 16, 20, 24)) abline(v = x + 0.5, col = 'grey')
for (x in c(4, 8, 12, 16, 20, 24, 28)) text(x - 3 + 0.5, 1.12, paste('Actor', x/4))
abline(h = 0.5, lty = 2, col = 'grey')
# savefig('model.jpg', width = 10, height = 3)
options(repr.plot.width = 7, repr.plot.height = 5)

These results should convey two things:
1. That the differences in handedness among chimps is the strongest effect
2. That the model as written assumes very little effect when the partner is added (filled circles)

We'll revisit this model later on when we look at multilevel models.

# Proportional odds

The results above give us absolute probabilites for each chimp pulling the left lever when the partner is present, but what about the relative change? These are called proportional odds and can be calculated very simply with a per unit increase in $x$, in other words the coefficient for the addition of a partner:

$$
\frac{exp(\beta_0+\beta_1(x_i+1))}{exp(\beta_0+\beta_1(x_i))}
$$

which, with a bit of algebra reduces down to

$$
\frac{exp(\beta_0)exp(\beta_1 x_i)exp(\beta_1)}{exp(\beta_0)exp(\beta_1 x_i)} = exp(\beta_1)
$$

For the chimps model, adding a partner is given by the difference between coefficients estimated in the treatment node of our model:

In [ ]:
# Right hand
RHpo <- exp(t3 - t1)
hist(RHpo, xlab = "", main = paste('Proportional odds (R) =', round(mean(RHpo), 2)))
abline(v = 1, col = 'red', lwd = 4)
# savefig('m1RTpo.jpg')

In [ ]:
# Left hand
LHpo <- exp(t4 - t2)
hist(LHpo, xlab = "", main = paste('Proportional odds (L) =', round(mean(LHpo), 2)))
abline(v = 1, col = 'red', lwd = 4)
# savefig('m1LTpo.jpg')

So in both cases, adding a partner to a particular side reduces the odds of pulling that lever.

# Aggregated binomial

While these data analyzsed so far are raw values for pulling the left lever (0/1), binomials are about numbers of successes in a given number of trials. As such, provided there is nothing special about the order of things, we can condense the data into a table of successes:

In [ ]:
# Add side covariate
cdata$side <- c('right', 'left')[cdata$prosoc_left + 1]
# Label treatments
cdata$treatment <- c('RN', 'LN', 'RP', 'LP')[cdata$treatment]
# Partner present
cdata$partner <- cdata$condition

In [ ]:
head(cdata, 5)

In [ ]:
cdata2 <- aggregate(list(pulled_left = cdata$pulled_left),
                    by = list(side = cdata$side, actor = cdata$actor,
                              treatment = cdata$treatment, partner = cdata$partner),
                    FUN = sum)
# pandas' pivot_table sorts on its index columns, so sort the same way here
cdata2 <- cdata2[order(cdata2$side, cdata2$actor, cdata2$treatment, cdata2$partner), ]
rownames(cdata2) <- NULL

In [ ]:
head(cdata2, 5)

In [ ]:
# Grab data
# Number of left pulls
L <- cdata2$pulled_left
# Individual chimps
tmp <- indexall(cdata2$actor)
Actor <- tmp[[1]]; Ia <- tmp[[2]]
Chimp <- paste('Chimp', Actor)
nchimps <- length(Actor)
# Treatment
tmp <- indexall(cdata2$treatment)
Treatment <- tmp[[1]]; It <- tmp[[2]]
ntreat <- length(Treatment)

The data are now counts of successes out of 18 trials rather than 0/1 outcomes, so the likelihood becomes `binomial_logit()` - again taking the linear model on the logit scale - with the number of trials passed in as data.

In [ ]:
Chimps2_code <- "
data {
  int<lower=0> N;                              // Number of observations
  int<lower=1> nchimps;
  int<lower=1> ntreat;
  int<lower=1> trials;                         // Trials per actor x treatment
  array[N] int<lower=1, upper=nchimps> Ia;     // Chimp index
  array[N] int<lower=1, upper=ntreat> It;      // Treatment index
  array[N] int<lower=0> L;                     // Number of left pulls - response
}
parameters {
  vector[nchimps] Actor;        // Individual intercepts
  vector[ntreat] Treatment;     // Treatment effects
}
model {
  // Individual intercepts
  Actor ~ normal(0, 1.7);
  // Treatment effects
  Treatment ~ normal(0, 0.5);

  // Linear model and likelihood
  L ~ binomial_logit(trials, Actor[Ia] + Treatment[It]);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = binomial_logit_lpmf(L[i] | trials, Actor[Ia[i]] + Treatment[It[i]]);
}
"
Chimps2 <- cmdstan_model(write_stan_file(Chimps2_code))

In [ ]:
trace_c2 <- sample_stan(Chimps2, list(N = length(L), nchimps = nchimps, ntreat = ntreat,
                                      trials = 18, Ia = Ia, It = It, L = L))

In [ ]:
tmp2 <- trace_c2$summary(c("Actor", "Treatment"))
tmp2$Label <- c(Chimp, Treatment)
tmp2

In [ ]:
# bayesplot draws one set of posteriors at a time, so stack the two models' draws side by side
raw_a <- trace_c$draws("Actor", format = "matrix")
colnames(raw_a) <- paste('Raw', Chimp)
agg_a <- trace_c2$draws("Actor", format = "matrix")
colnames(agg_a) <- paste('Aggregated', Chimp)
mcmc_intervals(cbind(raw_a, agg_a)[, as.vector(rbind(1:nchimps, nchimps + 1:nchimps))])
# ggsave('chimps_models_actor.jpg', dpi = 300)

In [ ]:
# The aggregated treatments come out in a different order (LN, LP, RN, RP), so line them
# up with the raw model's R/N, L/N, R/P, L/P before comparing
Treatment_raw <- c('R/N', 'L/N', 'R/P', 'L/P')
raw_t <- trace_c$draws("Treatment", format = "matrix")
colnames(raw_t) <- paste('Raw', Treatment_raw)
agg_t <- trace_c2$draws("Treatment", format = "matrix")[, match(c('RN', 'LN', 'RP', 'LP'), Treatment)]
colnames(agg_t) <- paste('Aggregated', Treatment_raw)
mcmc_intervals(cbind(raw_t, agg_t)[, as.vector(rbind(1:ntreat, ntreat + 1:ntreat))]) + vline_0()
# ggsave('chimps_models_treat.jpg', dpi = 300)

In [ ]:
raw_loo <- trace_c$loo()
agg_loo <- trace_c2$loo()

In [ ]:
?waic

In [ ]:
raw_waic <- waic(trace_c$draws("log_lik", format = "matrix"))
agg_waic <- waic(trace_c2$draws("log_lik", format = "matrix"))

In [ ]:
raw_waic

In [ ]:
agg_waic$pointwise[, "waic"]

In [ ]:
tmp <- raw_loo
tmp

In [ ]:
tmp2 <- agg_loo
tmp2

In [ ]:
Actor[Ia[agg_loo$diagnostics$pareto_k > 0.5]]

In [ ]:
agg_loo$diagnostics$pareto_k > 0.5

In [ ]:
loofail <- cdata2[agg_loo$diagnostics$pareto_k > 0.5, ]
loofail

This is interesting, but it's hard to see why these particular datapoints are failing - let's plot them to see:

In [ ]:
# Set figure size
options(repr.plot.width = 10, repr.plot.height = 3)
plot(NA, xlim = c(0.5, 28.5), ylim = c(0, 1.15), xlab = "", ylab = "", xaxt = 'n')
oset <- 1
for (i in 1:7){
    # Grab chimp
    l <- L[Ia == i]
    t <- It[Ia == i]
    # Grab data for each treatment (aggregated treatments are ordered LN, LP, RN, RP)
    rn <- l[t == which(Treatment == 'RN')]/18
    ln <- l[t == which(Treatment == 'LN')]/18
    rp <- l[t == which(Treatment == 'RP')]/18
    lp <- l[t == which(Treatment == 'LP')]/18
    # Grab k value fails
    lf2 <- loofail[loofail$actor == Actor[i], ]
    # Plot rights
    lines(c(oset, oset + 2), c(rn, rp), col = 'blue')
    points(oset, rn, col = 'blue', bg = 'white', pch = 21)
    if ('RN' %in% lf2$treatment) points(oset, rn, col = 'red', bg = 'white', pch = 21)
    points(oset + 2, rp, col = 'blue', bg = 'blue', pch = 21)
    if ('RP' %in% lf2$treatment) points(oset + 2, rp, col = 'red', bg = 'red', pch = 21)
    # Plot lefts
    lines(c(oset + 1, oset + 3), c(ln, lp), col = 'blue')
    points(oset + 1, ln, col = 'blue', bg = 'white', pch = 21)
    if ('LN' %in% lf2$treatment) points(oset + 1, ln, col = 'red', bg = 'white', pch = 21)
    points(oset + 3, lp, col = 'blue', bg = 'blue', pch = 21)
    if ('LP' %in% lf2$treatment) points(oset + 3, lp, col = 'red', bg = 'red', pch = 21)
    oset <- oset + 4
}
for (x in c(4, 8, 12, 16, 20, 24)) abline(v = x + 0.5, col = 'grey')
for (x in c(4, 8, 12, 16, 20, 24, 28)) text(x - 3 + 0.5, 1.12, paste('Actor', x/4))
abline(h = 0.5, lty = 2, col = 'grey')
legend('bottomright', legend = c('Alone', 'Partner'), pch = 21, col = 'blue',
       pt.bg = c('white', 'blue'), bty = 'n', horiz = TRUE)
# savefig('aggdata.jpg', width = 10, height = 3)
options(repr.plot.width = 7, repr.plot.height = 5)

So by using the aggregrated case, we have collapsed the data by a factor of 18 (the actor by treatment observations), and this makes it harder to fit out of sample. Rather than doing 'leave one out' cross-validation, it's more like 'leave 18 out' cross validation.

# Aggregated admissions

Here is an example of aggregated regression from UC Berkeley, a classic example looking at gender bias in admissions. First, import the data 

In [ ]:
bdata <- read.csv('UCBadmit.csv', sep = ";")
bdata$gender <- bdata$applicant.gender
bdata

The question here is **is there systematic bias against female applicants at UC Berkeley?** If we look at the proportions admitted by sex

In [ ]:
sum(bdata$admit[bdata$gender == 'female'])/sum(bdata$applications[bdata$gender == 'female'])

In [ ]:
sum(bdata$admit[bdata$gender == 'male'])/sum(bdata$applications[bdata$gender == 'male'])

So this looks bad - 14% more accepted males to accepted females, and at liberal-old Berkeley! Let's build a model to estimate the level of this effect:

In [ ]:
# Response
A <- bdata$admit
N <- bdata$applications

# Sex
tmp <- indexall(bdata$gender)
Sex_ <- tmp[[1]]; Is <- tmp[[2]]

In [ ]:
Admit_code <- "
data {
  int<lower=0> Nobs;                         // Number of rows
  int<lower=1> nsex;
  array[Nobs] int<lower=1, upper=nsex> Is;   // Sex index
  array[Nobs] int<lower=0> N;                // Applications
  array[Nobs] int<lower=0> A;                // Admissions - response
}
parameters {
  vector[nsex] Sex;     // log-odds of admission by sex
}
model {
  // log-odds of admission by sex
  Sex ~ normal(0, 1.7);

  // Linear model and likelihood
  A ~ binomial_logit(N, Sex[Is]);
}
"
Admit <- cmdstan_model(write_stan_file(Admit_code))

In [ ]:
trace_ba <- sample_stan(Admit, list(Nobs = nrow(bdata), nsex = length(Sex_),
                                    Is = Is, N = N, A = A))

In [ ]:
tmp <- trace_ba$summary("Sex")
tmp$Label <- Sex_
tmp

Calculate the probability of admission by sex

In [ ]:
Sex_draws <- trace_ba$draws("Sex", format = "matrix")
colMeans(invlogit(Sex_draws))

Which is just what we saw with the raw values. What's the difference in probability?

In [ ]:
diff <- invlogit(Sex_draws[, 1]) - invlogit(Sex_draws[, 2])
mean(diff)

In [ ]:
hist(diff, xlab = "", xlim = range(c(0, diff)), main = 'Berkeley Male vs Female acceptance probability')
abline(v = 0, col = 'red', lwd = 4)
# savefig('rawrates.jpg')

So the 14% we saw earlier. Now let's push these back through the model and see how our predictions look (i.e. the **most important model check**):

In [ ]:
plot(NA, xlim = c(0.5, 12.5), ylim = c(0, 1), ylab = 'Acceptance rate', xlab = 'Department')
for (i in 0:5){
    # Male/female per department
    x <- 1 + 2*i

    # Plot data
    y1 <- bdata$admit[x]/bdata$applications[x]
    y2 <- bdata$admit[x + 1]/bdata$applications[x + 1]
    lines(c(x, x + 1), c(y1, y2), type = 'b', col = 'dodgerblue', lwd = 2, pch = 16)
    text(x + 0.25, (y1 + y2)/2 + 0.05, bdata$dept[x])

    # Model predictions male
    pmale <- invlogit(Sex_draws[, 1])
    ynew1 <- quantile(pmale, 0.5)
    ynew1_lo <- quantile(pmale, 0.05)
    ynew1_hi <- quantile(pmale, 0.95)
    # Model predictions female
    pfemale <- invlogit(Sex_draws[, 2])
    ynew2 <- quantile(pfemale, 0.5)
    ynew2_lo <- quantile(pfemale, 0.025)
    ynew2_hi <- quantile(pfemale, 0.975)

    points(c(x, x + 1), c(ynew1, ynew2), col = 'black', pch = 16)
    lines(c(x, x + 1), c(ynew1, ynew2), col = 'grey')
    lines(c(x, x), c(ynew1_lo, ynew1_hi), col = 'black')
    lines(c(x + 1, x + 1), c(ynew2_lo, ynew2_hi), col = 'black')
}
legend('topright', legend = c('Data', 'Model'), col = c('dodgerblue', 'black'),
       lwd = c(2, NA), pch = 16, bty = 'n')
# savefig('rawmodelfit.jpg')

These are pretty shithouse predictions - and of course this example is classic because it is an example of Simpson's paradox. The key being to include a covariate for department - departments have wildly differing admission rates and depending on the numbers of male and female applications to each department, we can see bias where none exists. Let's run another model that handles all this:

In [ ]:
# Department
tmp <- indexall(bdata$dept)
Dept_ <- tmp[[1]]; Id <- tmp[[2]]

In [ ]:
AdmitD_code <- "
data {
  int<lower=0> Nobs;                         // Number of rows
  int<lower=1> nsex;
  int<lower=1> ndept;
  array[Nobs] int<lower=1, upper=nsex> Is;   // Sex index
  array[Nobs] int<lower=1, upper=ndept> Id;  // Department index
  array[Nobs] int<lower=0> N;                // Applications
  array[Nobs] int<lower=0> A;                // Admissions - response
}
parameters {
  vector[nsex] Sex;            // log-odds of admission by sex
  vector[ndept] Department;    // log-odds of admission by department
}
model {
  // log-odds of admission by sex
  Sex ~ normal(0, 1.7);
  // log-odds of admission by department
  Department ~ normal(0, 1.7);

  // Linear model and likelihood
  A ~ binomial_logit(N, Sex[Is] + Department[Id]);
}
"
AdmitD <- cmdstan_model(write_stan_file(AdmitD_code))

In [ ]:
trace_bad <- sample_stan(AdmitD, list(Nobs = nrow(bdata), nsex = length(Sex_), ndept = length(Dept_),
                                      Is = Is, Id = Id, N = N, A = A))

In [ ]:
Sex_bd <- trace_bad$draws("Sex", format = "matrix")
colnames(Sex_bd) <- paste('Sex', Sex_)
Dept_bd <- trace_bad$draws("Department", format = "matrix")
colnames(Dept_bd) <- paste('Department', Dept_)
mcmc_intervals(cbind(Sex_bd, Dept_bd)) + vline_0()
# ggsave('rightmodel.jpg', dpi = 300)

So the difference between sexes disappears, and males are accepted at slightly lower rates. We can use our DAG-building skills to see why:

In [ ]:
UCB_dag <- dagitty("dag {
  S -> D
  D -> A
  S -> A
}")
coordinates(UCB_dag) <- list(x = c(S = 0, D = 1, A = 2), y = c(S = 0, D = -1, A = 0))
plot(UCB_dag)
# savefig('UCB_dag.jpg')

Let's see what the model sees now:

In [ ]:
plot(NA, xlim = c(0.5, 12.5), ylim = c(0, 1), ylab = 'Acceptance rate', xlab = 'Department')
for (i in 0:5){
    # Male/female per department
    x <- 1 + 2*i

    # Plot data
    y1 <- bdata$admit[x]/bdata$applications[x]
    y2 <- bdata$admit[x + 1]/bdata$applications[x + 1]
    lines(c(x, x + 1), c(y1, y2), type = 'b', col = 'dodgerblue', lwd = 2, pch = 16)
    text(x + 0.25, (y1 + y2)/2 + 0.05, bdata$dept[x])

    # Model parameters
    bmale <- mean(Sex_bd[, 1])
    bfemale <- mean(Sex_bd[, 2])
    bdept <- mean(Dept_bd[, i + 1])
    ynew1 <- invlogit(bmale + bdept)
    ynew2 <- invlogit(bfemale + bdept)

    points(c(x, x + 1), c(ynew1, ynew2), col = 'black', pch = 16)
    lines(c(x, x + 1), c(ynew1, ynew2), col = 'grey')
}
legend('topright', legend = c('Data', 'Model'), col = c('dodgerblue', 'black'),
       lwd = c(2, NA), pch = 16, bty = 'n')
# savefig('rightmodelfit.jpg')

Doing better, but that Department A suggests much higher acceptance rates for females, something to be modelled later perhaps.